# RFP Generation Module Colab Experiment

이 노트북은 새로 받은 `src/generation/rfp_generation.py` 모듈을 기준으로 generation을 다시 실행합니다.

핵심 방향:
- 690 retrieval JSONL은 고정합니다.
- Chroma/vector DB는 사용하지 않습니다.
- `rfp_generation.py`의 field-aware context package, intent plan, deterministic calculation, postprocess, review output 저장 로직을 사용합니다.
- 이번 설정은 `source_store=True`입니다. source_store 경로가 없으면 즉시 중단합니다.
- 사람이 볼 결과는 `llm_answer_review.html`, `review_samples.csv`, `metrics_summary.json`입니다.


In [8]:
# 1. Experiment config
from pathlib import Path

REPO_URL = 'https://github.com/beomsookim1020/chatbot.git'
REPO_BRANCH = 'colab-generation'
PROJECT_DIR = Path('/content/chatbot')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')
DRIVE_EXPERIMENT_ROOT = DRIVE_OUTPUT_ROOT / 'generation_rfp_module_experiments'

EXPERIMENT_ID = 'rfp_module_690_clean'
EXPERIMENT_NAME = 'rfp_module_690_field_aware_source_store'

PREDICTION_REL = Path(
    'outputs/predictions/96_dense_qdecomp_rrf_per75_docscore_mean3_targetaware30_max5_preserve3_relaxed_filter_kure_chroma_690_canonical.jsonl'
)
CHUNK_SIDECAR_REL = Path('indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl')
EVAL_REL = Path('data/eval/representative_wrong_30_eval_batch_format.csv')
SOURCE_STORE_REL = Path('data/source_store_v2_690.jsonl')

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
RUN_LIMIT = 0
EXPERIMENT_IDS = []

MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.0
TOP_P = 1.0

USE_SOURCE_STORE = True
REVIEW_FOCUS = False
RANDOM_SEED = 42

GENERATION_CONFIG = {
    'use_source_store': USE_SOURCE_STORE,
    'max_context_chars_fact': 9000,
    'max_context_chars_synthesis': 12000,
    'max_blocks_fact': 8,
    'max_blocks_synthesis': 12,
    'evidence_text_chars': 1100,
    'source_store_text_chars': 1400,
}

print('experiment:', EXPERIMENT_NAME)
print('prediction:', PREDICTION_REL)
print('chunks:', CHUNK_SIDECAR_REL)
print('eval:', EVAL_REL)
print('model:', MODEL_NAME)


experiment: rfp_module_690_field_aware_source_store
prediction: outputs/predictions/96_dense_qdecomp_rrf_per75_docscore_mean3_targetaware30_max5_preserve3_relaxed_filter_kure_chroma_690_canonical.jsonl
chunks: indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl
eval: data/eval/representative_wrong_30_eval_batch_format.csv
model: Qwen/Qwen2.5-3B-Instruct


In [9]:
# 2. Runtime setup: GPU, Drive, repo, packages
import shutil
import subprocess
import sys

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Colab GPU runtime이 아닙니다. Runtime > Change runtime type > GPU로 바꿔주세요.')
subprocess.run(['nvidia-smi'], check=True)

from google.colab import drive
drive.mount('/content/drive')

def run_cmd(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=True)

if (PROJECT_DIR / '.git').exists():
    run_cmd(['git', 'fetch', 'origin', REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(['git', 'checkout', REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, PROJECT_DIR])

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0', 'accelerate', 'sentencepiece', 'safetensors'
], check=True)

ROOT = str(PROJECT_DIR)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

MODULE_PATH = PROJECT_DIR / 'src/generation/rfp_generation.py'
if not MODULE_PATH.exists():
    raise FileNotFoundError(
        f'{MODULE_PATH}가 없습니다. rfp_generation.py를 repo에 추가한 뒤 GitHub에 push하고 다시 실행해주세요.'
    )

print('Runtime ready.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin colab-generation
$ git checkout colab-generation
$ git pull --ff-only origin colab-generation
Runtime ready.


In [10]:
# 3. Copy inputs from Drive
import shutil

DRIVE_EVAL = DRIVE_INPUT_ROOT / EVAL_REL
DRIVE_PREDICTIONS = DRIVE_INPUT_ROOT / PREDICTION_REL
DRIVE_CHUNKS = DRIVE_INPUT_ROOT / CHUNK_SIDECAR_REL
DRIVE_SOURCE_STORE = DRIVE_INPUT_ROOT / SOURCE_STORE_REL

# 입력 경로는 실험 재현성을 위해 정확히 일치해야 합니다. fallback 경로나 silent skip은 사용하지 않습니다.
required_inputs = [
    ('eval', DRIVE_EVAL),
    ('predictions', DRIVE_PREDICTIONS),
    ('chunks', DRIVE_CHUNKS),
]
if USE_SOURCE_STORE:
    required_inputs.append(('source_store', DRIVE_SOURCE_STORE))

missing = [f'{label}: {path}' for label, path in required_inputs if not path.exists()]
if missing:
    raise FileNotFoundError('Configured input path does not exist. Fix the exact Drive path before running:\n' + '\n'.join(missing))

LOCAL_EVAL = PROJECT_DIR / EVAL_REL
LOCAL_PREDICTIONS = PROJECT_DIR / PREDICTION_REL
LOCAL_CHUNKS = PROJECT_DIR / CHUNK_SIDECAR_REL
LOCAL_SOURCE_STORE = PROJECT_DIR / SOURCE_STORE_REL if USE_SOURCE_STORE else None

LOCAL_EVAL.parent.mkdir(parents=True, exist_ok=True)
LOCAL_PREDICTIONS.parent.mkdir(parents=True, exist_ok=True)
LOCAL_CHUNKS.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_EVAL, LOCAL_EVAL)
shutil.copy2(DRIVE_PREDICTIONS, LOCAL_PREDICTIONS)
shutil.copy2(DRIVE_CHUNKS, LOCAL_CHUNKS)

if USE_SOURCE_STORE and LOCAL_SOURCE_STORE:
    LOCAL_SOURCE_STORE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_SOURCE_STORE, LOCAL_SOURCE_STORE)

local_required = [('eval', LOCAL_EVAL), ('predictions', LOCAL_PREDICTIONS), ('chunks', LOCAL_CHUNKS)]
if USE_SOURCE_STORE:
    local_required.append(('source_store', LOCAL_SOURCE_STORE))
missing_local = [f'{label}: {path}' for label, path in local_required if not path or not path.exists()]
if missing_local:
    raise FileNotFoundError('Input copy failed. Missing local files:\n' + '\n'.join(missing_local))

print('eval:', LOCAL_EVAL)
print('predictions:', LOCAL_PREDICTIONS)
print('chunks:', LOCAL_CHUNKS)
print('source_store:', LOCAL_SOURCE_STORE)


eval: /content/chatbot/data/eval/representative_wrong_30_eval_batch_format.csv
predictions: /content/chatbot/outputs/predictions/96_dense_qdecomp_rrf_per75_docscore_mean3_targetaware30_max5_preserve3_relaxed_filter_kure_chroma_690_canonical.jsonl
chunks: /content/chatbot/indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl
source_store: /content/chatbot/data/source_store_v2_690.jsonl


In [11]:
# 4. Import rfp_generation module and prepare items
import csv
import json
import time
from datetime import datetime, timezone

from src.generator import HuggingFaceGenerator
from src.generation.rfp_generation import (
    build_context_package,
    build_prompt,
    enrich_generation_record,
    load_chunk_index,
    load_generation_input_rows,
    load_source_store_index,
    postprocess_answer,
    prepare_generation_items,
    read_csv_records,
    save_generation_outputs,
)

result_rows, context_rows = load_generation_input_rows(
    LOCAL_PREDICTIONS,
    LOCAL_PREDICTIONS,
    experiment_id=EXPERIMENT_ID,
)

eval_rows = read_csv_records(LOCAL_EVAL)
if not eval_rows:
    raise ValueError(f'Eval CSV has no rows: {LOCAL_EVAL}')

missing_eval_id_rows = [idx for idx, row in enumerate(eval_rows, 1) if not str(row.get('id') or '').strip()]
if missing_eval_id_rows:
    raise ValueError('Eval CSV has rows without id: ' + ', '.join(map(str, missing_eval_id_rows[:20])))
eval_id_order = [str(row.get('id')).strip() for row in eval_rows]
seen_eval_ids = set()
duplicate_eval_ids = []
for qid in eval_id_order:
    if qid in seen_eval_ids and qid not in duplicate_eval_ids:
        duplicate_eval_ids.append(qid)
    seen_eval_ids.add(qid)
if duplicate_eval_ids:
    raise ValueError('Eval CSV has duplicate ids: ' + ', '.join(duplicate_eval_ids[:20]))

eval_by_id = {str(row.get('id')): row for row in eval_rows}
eval_ids = set(eval_id_order)
if EXPERIMENT_IDS:
    unknown_requested = [qid for qid in EXPERIMENT_IDS if qid not in eval_ids]
    if unknown_requested:
        raise ValueError('EXPERIMENT_IDS contains ids not present in eval CSV: ' + ', '.join(unknown_requested))
    target_id_order = list(EXPERIMENT_IDS)
else:
    target_id_order = eval_id_order
if RUN_LIMIT and RUN_LIMIT > 0:
    target_id_order = target_id_order[:RUN_LIMIT]
target_ids = set(target_id_order)

prediction_ids = {str(row.get('id') or row.get('question_id')) for row in result_rows}
context_ids = {str(row.get('question_id') or row.get('id')) for row in context_rows}
missing_prediction_ids = [qid for qid in target_id_order if qid not in prediction_ids]
missing_context_ids = [qid for qid in target_id_order if qid not in context_ids]
if missing_prediction_ids or missing_context_ids:
    details = []
    if missing_prediction_ids:
        details.append('missing prediction ids: ' + ', '.join(missing_prediction_ids[:30]))
    if missing_context_ids:
        details.append('missing retrieved context ids: ' + ', '.join(missing_context_ids[:30]))
    raise ValueError('Eval/retrieval id mismatch. Fix the retrieval file or eval set before running.\n' + '\n'.join(details))

# 대표 오답 eval CSV에 들어있는 id만 generation 대상으로 사용합니다.
# RUN_LIMIT=0은 prediction 전체가 아니라 이 eval 파일 전체(현재 30개)를 의미합니다.
result_rows = [
    row for row in result_rows
    if str(row.get('id') or row.get('question_id')) in target_ids
]
context_rows = [
    row for row in context_rows
    if str(row.get('question_id') or row.get('id')) in target_ids
]
for row in result_rows:
    qid = str(row.get('id') or row.get('question_id'))
    eval_row = eval_by_id.get(qid, {})
    if eval_row:
        row['question'] = eval_row.get('question') or row.get('question')
        row['ground_truth_answer'] = eval_row.get('ground_truth_answer', '')
        row['ground_truth_docs'] = eval_row.get('ground_truth_docs', '')
        row['type'] = eval_row.get('type', '')
        row['difficulty'] = eval_row.get('difficulty', '')
        row['metadata_filter'] = eval_row.get('metadata_filter', '')

items = prepare_generation_items(
    result_rows,
    context_rows,
    experiment_id=EXPERIMENT_ID,
    sample_size=None,
    review_focus=REVIEW_FOCUS,
    random_seed=RANDOM_SEED,
)
items_by_id = {str(item.get('question_id')): item for item in items}
items = [items_by_id[qid] for qid in target_id_order if qid in items_by_id]
if len(items) != len(target_id_order):
    missing_item_ids = [qid for qid in target_id_order if qid not in items_by_id]
    raise ValueError('Prepared item count mismatch. Missing item ids: ' + ', '.join(missing_item_ids[:30]))

chunk_ids = {
    str(context.get('chunk_id'))
    for item in items
    for context in item.get('retrieved_contexts', [])
    if context.get('chunk_id')
}
source_files = {
    str(context.get('source_file') or context.get('filename'))
    for item in items
    for context in item.get('retrieved_contexts', [])
    if context.get('source_file') or context.get('filename')
}

chunk_index = load_chunk_index(
    LOCAL_CHUNKS,
    chunk_ids=chunk_ids or None,
    source_files=source_files or None,
)
if not chunk_index:
    raise ValueError(f'Chunk index is empty. Check chunk sidecar path and chunk ids: {LOCAL_CHUNKS}')

source_store_index = load_source_store_index(
    LOCAL_SOURCE_STORE,
    enabled=bool(USE_SOURCE_STORE and LOCAL_SOURCE_STORE and LOCAL_SOURCE_STORE.exists()),
)

print('result rows:', len(result_rows))
print('context rows:', len(context_rows))
print('items:', len(items))
print('chunk index:', len(chunk_index))
print('source_store index:', len(source_store_index))
print('first ids:', [item.get('question_id') for item in items[:5]])


result rows: 30
context rows: 118
items: 30
chunk index: 13685
source_store index: 101737
first ids: ['Q006', 'Q017', 'Q020', 'Q031', 'Q123']


In [12]:
# 5. Dry-run context package preview
if not items:
    raise RuntimeError('No generation items selected.')

preview_item = items[0]
preview_package = build_context_package(
    preview_item['question'],
    preview_item['retrieved_contexts'],
    chunk_index=chunk_index,
    source_store_index=source_store_index,
    use_source_store=bool(USE_SOURCE_STORE and source_store_index),
    config=GENERATION_CONFIG,
)
preview_messages = build_prompt(preview_package)

print('question_id:', preview_item['question_id'])
print('question:', preview_item['question'])
print('analysis:', json.dumps(preview_package.get('question_analysis', {}), ensure_ascii=False, indent=2)[:3000])
print('evidence_blocks:', len(preview_package.get('evidence_blocks', [])))
print('context preview:')
print(preview_package.get('context_text', '')[:4000])
print('prompt roles:', [message['role'] for message in preview_messages])


question_id: Q006
question: 국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 위탁용역에서 응급실 섭외 지연 문제를 해결하기 위해 제시한 콜센터 구축 외에 함께 도입되는 통신 환경의 조건은 무엇인가요?
analysis: {
  "question_types": [
    "business_type"
  ],
  "period_subtypes": [],
  "is_multi_doc": false,
  "is_multi_intent": false,
  "needs_synthesis": false,
  "answer_type": "business_type",
  "intent_slots": [
    "general"
  ],
  "intent_plan": [
    {
      "intent_id": "I01",
      "intent": "general",
      "answer_section": "답변",
      "targets": [
        "국립중앙의료원 차세대 응급의료 상황관리시스템 구축 위탁용역"
      ],
      "target_policy": "single_target_preferred",
      "required_fact_types": [
        "document_summary"
      ],
      "preferred_chunk_types": [
        "text",
        "table",
        "fact_candidates"
      ],
      "requires_computation": false,
      "requires_all_targets": false,
      "classification_signals": []
    }
  ],
  "target_slots": [
    {
      "target_label": "국립중앙의료원 차세대 응급의료 상황관리시스템 구축 위탁용역",
      "issuer_hint": "",
      "pro

In [13]:
# 6. Run generation
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
LOCAL_OUTPUT_DIR = PROJECT_DIR / 'outputs/generation_rfp_module' / RUN_TIMESTAMP
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

generator = HuggingFaceGenerator(
    model_name=MODEL_NAME,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
)

def run_messages(messages):
    system_prompt = ''
    user_prompt = ''
    for message in messages:
        if message.get('role') == 'system':
            system_prompt = message.get('content', '')
        elif message.get('role') == 'user':
            user_prompt = message.get('content', '')
    return generator.generate_prompt(user_prompt, system_prompt=system_prompt or None)

records = []
for index, item in enumerate(items, 1):
    started = time.perf_counter()
    context_package = build_context_package(
        item['question'],
        item['retrieved_contexts'],
        chunk_index=chunk_index,
        source_store_index=source_store_index,
        use_source_store=bool(USE_SOURCE_STORE and source_store_index),
        config=GENERATION_CONFIG,
    )
    messages = build_prompt(context_package)
    raw_text = run_messages(messages)
    answer = postprocess_answer(raw_text, context_package)
    generation_ms = int((time.perf_counter() - started) * 1000)
    record = enrich_generation_record(
        answer,
        item,
        context_package,
        generation_ms=generation_ms,
        model_name=MODEL_NAME,
        experiment_name=EXPERIMENT_NAME,
        run_timestamp=RUN_TIMESTAMP,
    )
    record['_raw_text'] = raw_text
    records.append(record)
    print(
        f"{index}/{len(items)} {item['question_id']} type={record.get('answer_type')} "
        f"status={record.get('answer_status')} tags={record.get('_failure_tags')}"
    )

print('Local output dir:', LOCAL_OUTPUT_DIR)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

1/30 Q006 type=unknown status=not_found_in_context tags=[]
2/30 Q017 type=eligibility status=answered tags=[]
3/30 Q020 type=general status=answered tags=['target_doc_coverage_missing']
4/30 Q031 type=summary status=answered tags=['multi_intent_incomplete', 'target_doc_coverage_missing']
5/30 Q123 type=summary status=answered tags=[]
6/30 Q137 type=summary status=answered tags=['llm_hallucination_risk', 'source_numeric_missing']
7/30 Q001 type=budget status=insufficient_context tags=['llm_invalid_json', 'source_numeric_missing']
8/30 Q013 type=summary status=insufficient_context tags=['source_numeric_missing', 'target_doc_coverage_missing']
9/30 Q076 type=summary status=answered tags=['llm_invalid_json', 'multi_intent_incomplete']
10/30 Q104 type=summary status=not_found_in_context tags=['gt_expected_answer_but_model_not_found']
11/30 Q088 type=budget status=insufficient_context tags=['source_numeric_missing', 'citation_wrong_target']
12/30 Q007 type=budget status=insufficient_context 

In [14]:
# 7. Save outputs and copy to Drive
run_config = {
    'experiment_name': EXPERIMENT_NAME,
    'experiment_id': EXPERIMENT_ID,
    'model_name': MODEL_NAME,
    'prediction': str(PREDICTION_REL),
    'chunks': str(CHUNK_SIDECAR_REL),
    'eval': str(EVAL_REL),
    'run_limit': RUN_LIMIT,
    'use_source_store': USE_SOURCE_STORE,
    'generation_config': GENERATION_CONFIG,
    'run_timestamp': RUN_TIMESTAMP,
}

paths = save_generation_outputs(
    LOCAL_OUTPUT_DIR,
    records,
    run_config=run_config,
)

DRIVE_OUTPUT_DIR = DRIVE_EXPERIMENT_ROOT / RUN_TIMESTAMP
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in LOCAL_OUTPUT_DIR.glob('*'):
    if path.suffix.lower() in {'.jsonl', '.csv', '.html', '.json', '.md'}:
        shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)

print('Local output dir:', LOCAL_OUTPUT_DIR)
print('Drive output dir:', DRIVE_OUTPUT_DIR)
print('saved paths:')
for key, value in paths.items():
    print('-', key, value)


Local output dir: /content/chatbot/outputs/generation_rfp_module/20260527_072518
Drive output dir: /content/drive/MyDrive/chatbot_colab_outputs/generation_rfp_module_experiments/20260527_072518
saved paths:
- generated_answers /content/chatbot/outputs/generation_rfp_module/20260527_072518/generated_answers.jsonl
- review_samples /content/chatbot/outputs/generation_rfp_module/20260527_072518/review_samples.csv
- llm_answer_review /content/chatbot/outputs/generation_rfp_module/20260527_072518/llm_answer_review.csv
- llm_answer_review_html /content/chatbot/outputs/generation_rfp_module/20260527_072518/llm_answer_review.html
- metrics_summary /content/chatbot/outputs/generation_rfp_module/20260527_072518/metrics_summary.json
- failure_tags_summary /content/chatbot/outputs/generation_rfp_module/20260527_072518/failure_tags_summary.json
- ragas_eval_input /content/chatbot/outputs/generation_rfp_module/20260527_072518/ragas_eval_input.jsonl
- ragas_metrics_summary /content/chatbot/outputs/gen

## 결과 보는 순서

Drive 결과 경로:

`MyDrive/chatbot_colab_outputs/generation_rfp_module_experiments/{RUN_TIMESTAMP}/`

먼저 볼 파일:

1. `metrics_summary.json`: valid JSON, empty answer, numeric grounding, failure tag 요약
2. `failure_tags_summary.json`: 실패 태그별 예시
3. `llm_answer_review.html`: raw LLM text / parsed answer / final answer / GT 비교
4. `review_samples.csv`: 사람이 직접 `failure_type`, `review_memo`를 채울 검토용 CSV
5. `generated_answers.jsonl`: 전체 record와 context diagnostics
